# A1 (Pro版)：Cut-in 加塞场景 —— AEB 自动紧急制动算法闭环测试
# A1 (Pro): Cut-in Scenario — AEB Closed-Loop Simulation Test

> 💡 **版本说明**：本 Notebook 为 **A1 教学增强版 (Pro)**，重构了认知与操作路径：
> 1. **开箱即跑**：去除冗余终端命令，自动装载预置场景，零多余操作；
> 2. **直观场景**：新增 **2D 双车道路面鸟瞰图（BEV 关键帧抓拍）**，直观看见加塞避险过程；
> 3. **调参对比实验室**：一键运行 2.0s / 3.0s / 4.0s 灵敏度对比实验，同图叠画制动曲线；
> 4. **官方评分闭环**：仿真结果自动落盘，并即时调用平台评分器输出成绩单。

---

## 阶段一：3 分钟工程原理速览 | Principle & Background

### 1.1 什么是 Cut-in 加塞场景？
**加塞（Cut-in）**是高速路与城市快速路最高发的碰撞工况之一：相邻车道的目标车（Target）突然强行变道进入自车（Ego）正前方，留给驾驶员的反应时间极其短暂。Euro NCAP（欧洲新车安全评鉴协会）在 2023 规程中将其列为 AEB 关键测试项目。

### 1.2 TTC（碰撞时间）物理模型
$$\text{TTC} = \frac{d_{rel}}{v_{rel}} = \frac{x_{target} - x_{ego} - L_{veh}}{v_{ego} - v_{target}}$$
* **$\text{TTC} \le 0$ 或 $\infty$**：两车正在拉开距离或前车更快，无追尾危险；
* **$\text{TTC} < 2.5s$**：量产车系统通常在此阶段发出声光预警（FCW）；
* **$\text{TTC} < 1.5s$**：进入紧急危险区，必须全力介入制动（AEB），规避追尾。

### 1.3 核心矛盾：灵敏度 vs 误触发
* **阈值设得太大（如 4.0s）**：加塞车还没完全切入就一脚刹死，极易引起后方车辆追尾（幽灵刹车）；
* **阈值设得太小（如 2.0s）**：制动距离不足，刹不住导致碰撞。


In [1]:
# 确保 labs 目录进入 sys.path
import sys, os
for p in ['/home/jovyan/labs', os.path.abspath('../labs'), os.path.abspath('labs')]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)
        break

import matplotlib.pyplot as plt
from common.diagrams import ttc_principle, aeb_block_diagram
from common.vehicle_library import load_vehicle

# 车型参数加载（链1：车型设计 -> 智能驾驶）
veh_info = load_vehicle('bev_sedan')
print(f"✓ 实验车型: {veh_info['name']} (质量: {veh_info['mass_kg']} kg)")

# 1. 展示 TTC 计算数学原理
fig_ttc = ttc_principle()
plt.show()

# 2. 展示 AEB 系统四级闭环框图（感知 -> TTC计算 -> 决策逻辑 -> 制动执行）
fig_aeb = aeb_block_diagram()
plt.show()

print("✓ 原理示意图加载完成")


✓ 实验车型: 标准纯电轿车 (质量: 1800 kg)
✓ 原理示意图加载完成


## 阶段二：AEB 算法逻辑拆解 | Controller Implementation

我们继承平台标准基类 `BaseController`，编写一个极简且鲁棒的 TTC 紧急制动控制器：
* **输入 `observation`**：
  * `obs['ego']`：自车状态 `{'x', 'y', 'v', 'a', 'yaw'}`
  * `obs['targets']`：目标车列表 `[{'id', 'x', 'y', 'v'}, ...]`
* **输出 `action`**：
  * `action['target_a']`：期望纵向加速度（$m/s^2$）
  * `action['steer_angle']`：前轮转角（保持 $0.0$，纯纵向控制）


In [2]:
from common.controller import BaseController

class MyAEBController(BaseController):
    """基于 TTC 的紧急制动控制器"""

    def __init__(self, config=None):
        super().__init__(config)
        # 默认 TTC 触发阈值设为 3.0 秒，制动减速度为 -5.0 m/s^2
        self.ttc_threshold = self.config.get('ttc_threshold', 3.0)
        self.brake_accel = self.config.get('brake_accel', -5.0)

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        
        # 若感知视野内没有目标车，保持匀速
        if not targets:
            return {'target_a': 0.0, 'steer_angle': 0.0}

        # 1. 查找纵向距离最近的目标车
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        
        # 2. 计算净车距 (减去车身有效长度 4.5m) 与 相对速度
        rel_dist = abs(lead['x'] - ego['x']) - 4.5
        rel_v = ego['v'] - lead['v']

        # 3. 计算 TTC (若前车更快或相对远离，TTC 为无穷大)
        if rel_v <= 0.1:
            ttc = float('inf')
        else:
            ttc = rel_dist / rel_v

        # 4. 决策制动：当 TTC 小于预设阈值时触发强力制动
        if ttc < self.ttc_threshold:
            target_a = self.brake_accel
        else:
            target_a = 0.0  # 正常巡航，不施加制动

        return {'target_a': target_a, 'steer_angle': 0.0}

print('✓ AEB 控制器定义完成！')


✓ AEB 控制器定义完成！


## 阶段三：运行闭环仿真 | Run Simulation

在底层的 **esmini (v3.10.0)** 真实仿真引擎中：
1. 目标车（Target）从左车道以 $12 m/s$ 巡航，并在 $t=5.0s$ 执行变道切入自车前方；
2. 自车（Ego）以初速 $14 m/s$（约 $50 km/h$）直行行驶；
3. 你的控制器每步接收车辆位姿，输出制动动作并实时回传给物理模型。


In [3]:
from common.esmini_env import EsminiEnv

def run_aeb_simulation(ttc_threshold=3.0, brake_accel=-5.0, steps=400):
    """运行一次完整的 AEB 闭环仿真，返回详细轨迹数据与核心指标"""
    # 自动探测场景文件路径
    scenario_paths = [
        '/home/jovyan/labs/scenarios/A_esmini/A1_cutin/cutin.xosc',
        'labs/scenarios/A_esmini/A1_cutin/cutin.xosc',
        '../labs/scenarios/A_esmini/A1_cutin/cutin.xosc'
    ]
    scenario_file = next((p for p in scenario_paths if os.path.exists(p)), scenario_paths[0])
    road_file = scenario_file.replace('.xosc', '.xodr')

    controller = MyAEBController(config={
        'ttc_threshold': ttc_threshold,
        'brake_accel': brake_accel
    })
    
    env = EsminiEnv(scenario=scenario_file, road=road_file, dt=0.05)
    obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=14.0)
    controller.reset()

    traj = []
    min_ttc, is_collision, min_gap = float('inf'), False, float('inf')

    for step in range(steps):
        action = controller.step(obs)
        obs, done, info = env.step(action)
        t = obs['sim_time']
        ego = obs['ego']
        
        # 记录关键目标车信息
        tgt_x, tgt_y, tgt_v = None, None, None
        ttc = float('inf')
        for tgt in obs.get('targets', []):
            rel_d = abs(tgt['x'] - ego['x']) - 4.5
            min_gap = min(min_gap, rel_d)
            rel_v = ego['v'] - tgt['v']
            if rel_v > 0.1:
                ttc = rel_d / rel_v
                min_ttc = min(min_ttc, ttc)
            if rel_d < 0.5:
                is_collision = True
            tgt_x, tgt_y, tgt_v = tgt['x'], tgt['y'], tgt['v']

        traj.append({
            't': t, 'ego_x': ego['x'], 'ego_y': ego['y'], 'ego_v': ego['v'],
            'tgt_x': tgt_x, 'tgt_y': tgt_y, 'tgt_v': tgt_v,
            'ttc': ttc, 'target_a': action['target_a']
        })
        if done or is_collision:
            break

    env.close()
    return {
        'ttc_threshold': ttc_threshold,
        'brake_accel': brake_accel,
        'min_ttc': min_ttc,
        'min_gap': min_gap,
        'is_collision': is_collision,
        'trajectory': traj
    }

# 运行基线实验（TTC 阈值 = 3.0s）
baseline_res = run_aeb_simulation(ttc_threshold=3.0, brake_accel=-5.0)

print(f"✓ 闭环仿真完成！步数: {len(baseline_res['trajectory'])} 步 (时长: {baseline_res['trajectory'][-1]['t']:.1f}s)")
print(f"  • 最小 TTC: {baseline_res['min_ttc']:.2f} s （通过线 ≥ 1.5s）")
print(f"  • 最小净间距: {baseline_res['min_gap']:.2f} m")
print(f"  • 是否发生碰撞: {'💥 发生碰撞!' if baseline_res['is_collision'] else '🛡️ 安全无碰撞'}")


esmini GIT REV: v3.10.0-0-06065f92
esmini GIT TAG: v3.10.0
esmini GIT BRANCH: tags/v3.10.0^0
esmini BUILD VERSION: 6442
[] [info] Ignoring missing default config: /opt/esmini/esmini/bin/../config.yml
[] [info] Loaded cutin.xosc
Unable to open log file log.txt
Unable to open log file log.txt
[2026-10-26 17:45:20]
Unable to open log file log.txt
[] [info] Player options: --osc cutin.xosc --camera_mode orbit --fixed_timestep 0.05 --ground_plane auto --headless --info_text 1 --logfile_path log.txt --log_level info --text_scale 1.0 --traj_filter 0.1 --light_mode auto 
Unable to open log file log.txt
[] [info] Run simulation decoupled from realtime, with fixed timestep: 0.05
Unable to open log file log.txt
[] [info] Generated seed 1113530359
Unable to open log file log.txt
[] [info] Loading cutin.xosc (v1.1)
Unable to open log file log.txt
[] [info] 0 variables
Unable to open log file log.txt
[] [info] Loaded OpenDRIVE: cutin.xodr
Unable to open log file log.txt
[] [info] Non zero vehicle re

[20.000] [info] Closing
Unable to open log file log.txt
✓ 闭环仿真完成！步数: 400 步 (时长: 20.0s)
  • 最小 TTC: 2.96 s （通过线 ≥ 1.5s）
  • 最小净间距: 0.74 m
  • 是否发生碰撞: 🛡️ 安全无碰撞


## 阶段四：直观场景可视化 | Multi-Angle Visualization

为了让大家直观“看”到自动驾驶车辆的加塞避险过程，我们绘制了：
1. **2D 双车道路面鸟瞰图（BEV Snapshot）**：展示 3 个关键时刻自车（蓝车）与加塞目标车（红车）的实际空间位置；
2. **速度变化与制动曲线**：直观展示自车在加塞触发瞬间的减速过程；
3. **TTC 实时动态演变曲线**：观察 TTC 如何从安全区下降至阈值并被 AEB 及时挽回。

> 注：鸟瞰图中车道横向位置为示意，车辆纵向位置与时刻均取自真实仿真轨迹。


In [4]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import HTML, display
def plot_simulation_dashboard(res):
    traj = res['trajectory']
    times = [d['t'] for d in traj]
    ego_speeds = [d['ego_v'] for d in traj]
    ttcs = [min(d['ttc'], 10.0) for d in traj]
    fig = plt.figure(figsize=(13, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2])
    # 1. 上半部分：2D 双车道鸟瞰路面快照 (BEV Multi-Frame Snapshot)
    ax_bev = fig.add_subplot(gs[0, :])
    ax_bev.set_facecolor('#262626')
    ax_bev.axhline(0, color='white', linewidth=2)
    ax_bev.axhline(3.5, color='#fbbf24', linestyle='--', linewidth=1.5, alpha=0.9)
    ax_bev.axhline(7.0, color='white', linewidth=2)
    snapshot_times = [0.0, 5.0, min(7.5, times[-1])]
    labels = ['t=0.0s Init Cruise', 't=5.0s Target Cut-in', f't={min(7.5, times[-1]):.1f}s Safe Stop']
    colors_ego = ['#60a5fa', '#3b82f6', '#1d4ed8']
    colors_tgt = ['#f87171', '#ef4444', '#b91c1c']
    for s_idx, target_t in enumerate(snapshot_times):
        frame = min(traj, key=lambda d: abs(d['t'] - target_t))
        ego_x = frame['ego_x']
        tgt_x = frame['tgt_x'] if frame['tgt_x'] else 0
        tgt_y_m = 5.2 if frame['t'] < 5.0 else 1.75
        ax_bev.add_patch(patches.Rectangle((ego_x, 0.75), 4.8, 2.0, color=colors_ego[s_idx], ec='white', lw=1))
        ax_bev.text(ego_x + 2.4, 1.75, "Ego", ha='center', va='center', color='white', fontsize=7.5, fontweight='bold')
        ax_bev.add_patch(patches.Rectangle((tgt_x, tgt_y_m - 1.0), 4.8, 2.0, color=colors_tgt[s_idx], ec='white', lw=1))
        ax_bev.text(tgt_x + 2.4, tgt_y_m, "Target", ha='center', va='center', color='white', fontsize=7.5, fontweight='bold')
        ax_bev.text(min(ego_x, tgt_x), 6.1, labels[s_idx], color='#e2e8f0', fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.2", fc="#334155", ec="#64748b", alpha=0.85))
    ax_bev.set_xlim(0, max(traj[-1]['tgt_x'] + 10, 150))
    ax_bev.set_ylim(-0.5, 7.5)
    ax_bev.set_yticks([1.75, 5.25])
    ax_bev.set_yticklabels(['Lane 1 (Ego Lane)', 'Lane 2 (Target Lane)'], fontsize=9)
    ax_bev.set_xlabel('Road Position X (m)', fontsize=9.5)
    ax_bev.set_title("Cut-in Scenario Bird's-Eye View (BEV Multi-Stage Snapshot)", fontsize=11, fontweight='bold')
    ax_bev.grid(True, axis='x', alpha=0.2, linestyle=':')
    # 2. 下半部分左图：速度随时间变化曲线
    ax_v = fig.add_subplot(gs[1, 0])
    ax_v.plot(times, ego_speeds, color='#155eef', lw=2.5, label='Ego Speed (m/s)')
    ax_v.axhline(0, color='#ef4444', linestyle='--', alpha=0.7, label='Stopped')
    ax_v.axvline(5.0, color='#f59e0b', linestyle=':', label='t=5.0s Cut-in Event')
    ax_v.set_xlabel('Time (s)', fontsize=9.5)
    ax_v.set_ylabel('Speed (m/s)', fontsize=9.5)
    ax_v.set_title('AEB Speed Profile & Deceleration Response', fontsize=11, fontweight='bold')
    ax_v.grid(True, alpha=0.3)
    ax_v.legend(fontsize=8.5)
    # 3. 下半部分右图：实时 TTC 变化曲线
    ax_ttc = fig.add_subplot(gs[1, 1])
    ax_ttc.plot(times, ttcs, color='#10b981', lw=2.5, label='Real-time TTC')
    ax_ttc.axhline(res['ttc_threshold'], color='#ef4444', linestyle='--', label=f"AEB Threshold ({res['ttc_threshold']}s)")
    ax_ttc.axhline(1.5, color='#f59e0b', linestyle=':', label='NCAP Safe Line (1.5s)')
    ax_ttc.set_xlabel('Time (s)', fontsize=9.5)
    ax_ttc.set_ylabel('TTC (s)', fontsize=9.5)
    ax_ttc.set_title('Time-to-Collision (TTC) Dynamic Curve', fontsize=11, fontweight='bold')
    ax_ttc.set_ylim(0, 8)
    ax_ttc.grid(True, alpha=0.3)
    ax_ttc.legend(fontsize=8.5)
    plt.tight_layout()
    plt.show()
def render_interactive_player(res):
    """生成并渲染轻量级原生 HTML5/Canvas 交互式 2D 动态回放器"""
    traj = res['trajectory']
    step = max(1, len(traj) // 100)
    sampled = traj[::step]
    if traj and sampled[-1] != traj[-1]:
        sampled.append(traj[-1])
    data_json = json.dumps([{
        't': round(d['t'], 2),
        'ego_x': round(d['ego_x'], 1),
        'ego_v': round(d['ego_v'], 1),
        'tgt_x': round(d['tgt_x'] if d['tgt_x'] is not None else 0.0, 1),
        'tgt_y': round(d['tgt_y'] if d['tgt_y'] is not None else -1.5, 2),
        'gap': round(max(0.0, (d['tgt_x'] - d['ego_x'] - 4.5) if d['tgt_x'] is not None else 0.0), 1),
        'ttc': round(min(d['ttc'], 99.9), 1),
        'a': round(d['target_a'], 1)
    } for d in sampled])
    html_code = f"""
    <div id="a1_player_wrap" style="background: #1e293b; border-radius: 10px; padding: 14px; font-family: system-ui, sans-serif; color: #f8fafc; max-width: 820px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.3); margin-top: 15px;">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
        <div style="font-weight: bold; font-size: 14px; display: flex; align-items: center; gap: 8px;">
          <span style="color: #38bdf8;">🚗 2D 场景动态交互回放器</span>
          <span id="a1_status" style="background: #334155; padding: 2px 8px; border-radius: 4px; font-size: 11px;">准备就绪</span>
        </div>
        <div style="font-size: 12px; color: #94a3b8; font-family: monospace;">
          t: <span id="a1_t" style="color: #f8fafc; font-weight: bold;">0.00</span>s | 
          v: <span id="a1_v" style="color: #38bdf8; font-weight: bold;">0.0</span> m/s | 
          间距: <span id="a1_gap" style="color: #fbbf24; font-weight: bold;">0.0</span> m | 
          TTC: <span id="a1_ttc" style="color: #4ade80; font-weight: bold;">0.0</span> s
        </div>
      </div>
      <canvas id="a1_canvas" width="790" height="120" style="width: 100%; height: 120px; background: #0f172a; border-radius: 6px; display: block;"></canvas>
      <div style="display: flex; align-items: center; gap: 10px; margin-top: 10px;">
        <button id="a1_btn_play" style="background: #2563eb; color: white; border: none; padding: 5px 14px; border-radius: 5px; cursor: pointer; font-size: 12px; font-weight: bold;">▶ 播放</button>
        <button id="a1_btn_reset" style="background: #475569; color: white; border: none; padding: 5px 12px; border-radius: 5px; cursor: pointer; font-size: 12px;">⏪ 复位</button>
        <input id="a1_slider" type="range" min="0" max="{len(sampled)-1}" value="0" style="flex: 1; accent-color: #38bdf8; cursor: pointer;">
        <select id="a1_speed" style="background: #334155; color: #f8fafc; border: 1px solid #475569; padding: 3px 6px; border-radius: 4px; font-size: 11px;">
          <option value="0.5">0.5x</option>
          <option value="1" selected>1.0x</option>
          <option value="2">2.0x</option>
        </select>
      </div>
      <script>
      (function() {{
        const data = {data_json};
        const canvas = document.getElementById('a1_canvas');
        if (!canvas) return;
        const ctx = canvas.getContext('2d');
        const slider = document.getElementById('a1_slider');
        const btnPlay = document.getElementById('a1_btn_play');
        const btnReset = document.getElementById('a1_btn_reset');
        const speedSel = document.getElementById('a1_speed');
        const txtStatus = document.getElementById('a1_status');
        const txtT = document.getElementById('a1_t');
        const txtV = document.getElementById('a1_v');
        const txtGap = document.getElementById('a1_gap');
        const txtTTC = document.getElementById('a1_ttc');
        let curIdx = 0;
        let isPlaying = false;
        let timer = null;
        function renderFrame(idx) {{
          curIdx = idx;
          slider.value = idx;
          const d = data[idx];
          txtT.textContent = d.t.toFixed(2);
          txtV.textContent = d.ego_v.toFixed(1);
          txtGap.textContent = d.gap.toFixed(1);
          txtTTC.textContent = d.ttc >= 50 ? '∞' : d.ttc.toFixed(1);
          if (d.ego_v <= 0.1 && d.t > 6) {{
            txtStatus.textContent = '✅ 安全避险刹停';
            txtStatus.style.background = '#059669';
          }} else if (d.a < -2.0) {{
            txtStatus.textContent = '🚨 AEB 紧急制动介入中!';
            txtStatus.style.background = '#dc2626';
          }} else if (d.t >= 5.0) {{
            txtStatus.textContent = '⚠️ 目标车切入车道';
            txtStatus.style.background = '#d97706';
          }} else {{
            txtStatus.textContent = '🚗 巡航行驶中';
            txtStatus.style.background = '#334155';
          }}
          const W = canvas.width, H = canvas.height;
          ctx.clearRect(0, 0, W, H);
          ctx.fillStyle = '#1e293b';
          ctx.fillRect(0, 15, W, 90);
          ctx.strokeStyle = '#e2e8f0';
          ctx.lineWidth = 2;
          ctx.beginPath();
          ctx.moveTo(0, 15); ctx.lineTo(W, 15);
          ctx.moveTo(0, 105); ctx.lineTo(W, 105);
          ctx.stroke();
          ctx.strokeStyle = '#f59e0b';
          ctx.setLineDash([12, 10]);
          ctx.lineWidth = 1.5;
          ctx.beginPath();
          ctx.moveTo(0, 60); ctx.lineTo(W, 60);
          ctx.stroke();
          ctx.setLineDash([]);
          ctx.fillStyle = '#64748b';
          ctx.font = '10px sans-serif';
          ctx.fillText('Lane 2 (左侧车道)', 10, 32);
          ctx.fillText('Lane 1 (自车所在车道)', 10, 77);
          const camX = (d.ego_x + d.tgt_x) / 2;
          const scale = 3.6;
          const offsetX = W / 2 - camX * scale;
          function toScreenX(x) {{ return x * scale + offsetX; }}
          const egoScreenX = toScreenX(d.ego_x);
          const egoScreenY = 75;
          ctx.fillStyle = '#2563eb';
          ctx.strokeStyle = '#93c5fd';
          ctx.lineWidth = 1.5;
          ctx.fillRect(egoScreenX - 22, egoScreenY - 10, 44, 20);
          ctx.strokeRect(egoScreenX - 22, egoScreenY - 10, 44, 20);
          if (d.a < -1.0) {{
            ctx.fillStyle = '#ff0000';
            ctx.shadowColor = '#ff0000';
            ctx.shadowBlur = 8;
            ctx.fillRect(egoScreenX - 24, egoScreenY - 9, 3, 5);
            ctx.fillRect(egoScreenX - 24, egoScreenY + 4, 3, 5);
            ctx.shadowBlur = 0;
          }}
          ctx.fillStyle = '#ffffff';
          ctx.font = 'bold 10px sans-serif';
          ctx.fillText('Ego', egoScreenX - 10, egoScreenY + 3.5);
          const tgtScreenX = toScreenX(d.tgt_x);
          let tgtScreenY = 32;
          if (d.t >= 5.0) {{
            const cutinProgress = Math.min(1.0, (d.t - 5.0) / 1.5);
            tgtScreenY = 32 + (75 - 32) * cutinProgress;
          }}
          ctx.fillStyle = '#dc2626';
          ctx.strokeStyle = '#fca5a5';
          ctx.lineWidth = 1.5;
          ctx.fillRect(tgtScreenX - 22, tgtScreenY - 10, 44, 20);
          ctx.strokeRect(tgtScreenX - 22, tgtScreenY - 10, 44, 20);
          ctx.fillStyle = '#ffffff';
          ctx.font = 'bold 10px sans-serif';
          ctx.fillText('Target', tgtScreenX - 15, tgtScreenY + 3.5);
          if (tgtScreenX > egoScreenX) {{
            ctx.strokeStyle = '#fbbf24';
            ctx.lineWidth = 1.5;
            ctx.beginPath();
            ctx.moveTo(egoScreenX + 22, egoScreenY);
            ctx.lineTo(tgtScreenX - 22, egoScreenY);
            ctx.stroke();
            ctx.fillStyle = '#fbbf24';
            ctx.font = 'bold 11px sans-serif';
            ctx.fillText(d.gap.toFixed(1) + 'm', (egoScreenX + tgtScreenX) / 2 - 12, egoScreenY - 5);
          }}
        }}
        slider.addEventListener('input', (e) => {{
          renderFrame(parseInt(e.target.value));
        }});
        function playNext() {{
          if (!isPlaying) return;
          if (curIdx < data.length - 1) {{
            renderFrame(curIdx + 1);
            const speed = parseFloat(speedSel.value);
            timer = setTimeout(playNext, 50 / speed);
          }} else {{
            isPlaying = false;
            btnPlay.textContent = '▶ 播放';
          }}
        }}
        btnPlay.addEventListener('click', () => {{
          if (isPlaying) {{
            isPlaying = false;
            btnPlay.textContent = '▶ 播放';
            clearTimeout(timer);
          }} else {{
            if (curIdx >= data.length - 1) curIdx = 0;
            isPlaying = true;
            btnPlay.textContent = '⏸ 暂停';
            playNext();
          }}
        }});
        btnReset.addEventListener('click', () => {{
          isPlaying = false;
          btnPlay.textContent = '▶ 播放';
          clearTimeout(timer);
          renderFrame(0);
        }});
        renderFrame(0);
      }})();
      </script>
    </div>
    """
    display(HTML(html_code))
plot_simulation_dashboard(baseline_res)
render_interactive_player(baseline_res)

# ----------------------------------------------------
# 3. 🌐 [Phase 1 创新升级]：双视角数字孪生动态回放器（2D 画布：俯视 + 斜视角投影）
# ----------------------------------------------------
try:
    from common.visualizer3d import render_trajectory_replay
    _traj = baseline_res["trajectory"]
    _t = [d["t"] for d in _traj]
    _ego = {"x": [d["ego_x"] for d in _traj], "y": [d["ego_y"] for d in _traj], "v": [d["ego_v"] for d in _traj], "a": [d["target_a"] for d in _traj]}
    _tgt = {"name": "Target Cut-in", "color": "#FF5722", "x": [d["tgt_x"] for d in _traj], "y": [d["tgt_y"] for d in _traj], "v": [d["tgt_v"] for d in _traj]}
    _metrics = {"ttc": [d["ttc"] for d in _traj]}
    display(render_trajectory_replay(_t, _ego, targets=[_tgt], metrics=_metrics, title="A1 智驾场景 · Cut-in 避险数字孪生全景视界"))
except Exception as e:
    print(f"ℹ️ 数字孪生回放器加载跳过: {e}")


## 阶段五：参数调优实验室 | Parameter Exploration Lab

### 🔬 调优任务：灵敏度 vs 误触发
请思考并观察：改变 `ttc_threshold` 会对车辆制动产生什么影响？
* **设置 2.0s**：AEB 触发晚，制动距离是否足够？会不会刹不住？
* **设置 4.0s**：AEB 触发过早，前车尚未完全进入车道就急刹，是否会惊吓驾驶员并引起后车追尾？
* **设置 3.0s**：基线设定，安全性与舒适性的平衡。

下面代码将**一行运行 3 组参数**，并在同一张图上叠画对比，自动汇总测试表格：


In [5]:
def compare_ttc_thresholds(thresholds=[2.0, 3.0, 4.0]):
    print("=" * 65)
    print("  TTC 阈值   |   最小 TTC   |    最小间距    |     制动结果     |   是否碰撞  ")
    print("=" * 65)

    plt.figure(figsize=(10, 4))
    results = {}

    for th in thresholds:
        res = run_aeb_simulation(ttc_threshold=th)
        results[th] = res
        times = [d['t'] for d in res['trajectory']]
        speeds = [d['ego_v'] for d in res['trajectory']]
        status = "💥 发生碰撞" if res['is_collision'] else ("⚠️ 险些追尾" if res['min_gap'] < 2.0 else "✅ 安全避险")
        coll_str = "是" if res['is_collision'] else "否"

        print(f"{th:^10.1f}s | {res['min_ttc']:^10.2f}s | {res['min_gap']:^9.1f}m | {status:^10} | {coll_str:^8}")
        plt.plot(times, speeds, lw=2, label=f'TTC Threshold = {th:.1f}s (Min Gap: {res["min_gap"]:.1f}m)')

    print("=" * 65)
    plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Stopped')
    plt.xlabel('Time (s)', fontsize=9.5)
    plt.ylabel('Speed (m/s)', fontsize=9.5)
    plt.title('Braking Response Comparison across TTC Thresholds', fontsize=11, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=9)
    plt.show()
    return results

comp_results = compare_ttc_thresholds([2.0, 3.0, 4.0])

# ----------------------------------------------------
# 🎯 5.2 学生动手标定挑战 (Hands-on Calibration Challenge)
# ----------------------------------------------------
print("\n💡 [学生动手区] 寻找最优标定参数：")
print("尝试修改下面的参数，观察能否在满足最小 TTC >= 1.5s 的前提下，缩短制动距离：")
my_custom_ttc = 2.8      # 尝试调整：2.4 ~ 3.5 秒
my_custom_decel = -5.5    # 尝试调整：-4.5 ~ -7.0 m/s^2

custom_test = run_aeb_simulation(ttc_threshold=my_custom_ttc, brake_accel=my_custom_decel)
print(f"👉 自定义标定结果: 最小 TTC={custom_test['min_ttc']:.2f}s, 最小间隙={custom_test['min_gap']:.2f}m, 碰撞={custom_test['is_collision']}")



  TTC 阈值   |   最小 TTC   |    最小间距    |     制动结果     |   是否碰撞  
esmini GIT REV: v3.10.0-0-06065f92
esmini GIT TAG: v3.10.0
esmini GIT BRANCH: tags/v3.10.0^0
esmini BUILD VERSION: 6442
[] [info] Ignoring missing default config: /opt/esmini/esmini/bin/../config.yml
[] [info] Loaded cutin.xosc
Unable to open log file log.txt
Unable to open log file log.txt
[2026-10-26 17:45:20]
Unable to open log file log.txt
[] [info] Player options: --osc cutin.xosc --camera_mode orbit --fixed_timestep 0.05 --ground_plane auto --headless --info_text 1 --logfile_path log.txt --log_level info --text_scale 1.0 --traj_filter 0.1 --light_mode auto 
Unable to open log file log.txt
[] [info] Run simulation decoupled from realtime, with fixed timestep: 0.05
Unable to open log file log.txt
[] [info] Generated seed 1113530359
Unable to open log file log.txt


[] [info] Loading cutin.xosc (v1.1)
Unable to open log file log.txt
[] [info] 0 variables
Unable to open log file log.txt
[] [info] Loaded OpenDRIVE: cutin.xodr
Unable to open log file log.txt
[] [info] Non zero vehicle rear axle x offset: -1.40 found for EgoCar. Shift 3D model and adjust axles and bounding box accordingly.
Unable to open log file log.txt
[] [info] 3D model search paths attempted for file car_white.osgb:
Unable to open log file log.txt
[] [info]   car_white.osgb
Unable to open log file log.txt
[] [info]   .//./car_white.osgb
Unable to open log file log.txt
[] [info]   /opt/esmini/esmini/bin/../resources/models//./car_white.osgb
Unable to open log file log.txt
[] [info]   .//../models/./car_white.osgb
Unable to open log file log.txt
[] [warn] 3D model car_white.osgb not located, using specified file path
Unable to open log file log.txt
[] [info] Assigned 3D model car_white.osgb to vehicle EgoCar
Unable to open log file log.txt
[] [info] Add and deactivate new object "Eg

## 阶段六：官方评分与结果归档 | Auto-Grading & Results

平台通过自动评分器 `labs/common/evaluator.py` 检验学生的实验完成度：
* **无碰撞 (`collision == False`)**：权重 2/5；
* **最小 TTC 达标 (`min_ttc_s >= 1.5`)**：权重 3/5；
* **结果文件完整性**：20分。

下面代码将结果写入 `labs/data/results/A1_cutin_aeb.json`，并当场调用评分器输出你的官方成绩单！


In [6]:
import json
from common.evaluator import grade_metrics

# 1. 整理基线实验达标指标并落盘（供平台自动评分与教师看板采集）
save_dir = '/home/jovyan/labs/data/results'
record = {
    'experiment': 'A1_cutin_aeb',
    'metrics': {
        'min_ttc_s': round(baseline_res['min_ttc'], 2),
        'collision': baseline_res['is_collision'],
        'min_gap_m': round(baseline_res['min_gap'], 2)
    }
}

saved_ok = False
for target_dir in [save_dir, 'labs/data/results', 'results']:
    try:
        os.makedirs(target_dir, exist_ok=True)
        out_file = os.path.join(target_dir, 'A1_cutin_aeb.json')
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f"✓ 结果文件已成功归档至: {out_file}")
        saved_ok = True
        break
    except OSError:
        continue

if not saved_ok:
    print("ℹ️ 提示: 当前运行环境为只读挂载模式，跳过落盘，直接进入评测")

# 2. 调用平台官方评测器输出成绩单
eval_report = grade_metrics('A1_cutin_aeb', record['metrics'])

print("\n" + "=" * 50)
print(f"🏆 IEV CloudLab 官方评分报告: {eval_report['name']}")
print("=" * 50)
print(f"总得分: {eval_report['score']} / 100 分  ({'✅ 通过' if eval_report['passed'] else '❌ 未通过'})")
print("-" * 50)
for d in eval_report['details']:
    status_icon = "✓ 通过" if d['passed'] else "✗ 未达标"
    print(f"• 指标 [{d['metric']}]: 当前值 = {d['value']} ({d['desc']}) -> {status_icon}")
print("=" * 50)


ℹ️ 提示: 当前运行环境为只读挂载模式，跳过落盘，直接进入评测

🏆 IEV CloudLab 官方评分报告: Cut-in加塞避险
总得分: 100.0 / 100 分  (✅ 通过)
--------------------------------------------------
• 指标 [min_ttc_s]: 当前值 = 2.96 (最小TTC大于1.5s（AEB有效避险）) -> ✓ 通过
• 指标 [collision]: 当前值 = False (无碰撞) -> ✓ 通过


## 阶段七：工程师实战经验与进阶挑战 | Reflection & Challenges

### 7.1 量产工程师的调校思考
当你拿到基线实验“最小 TTC = 2.96s”的测试结果时，量产控制算法工程师会如何思考？
1. **安全裕度是否过大？**
   NCAP 评测线是 1.5s，我们实测是 2.96s，留了近 1.5s 的充裕余量。在实车标定中，工程师通常会将阈值略微收紧（如调至 2.5s），让制动稍晚一点点触发，以提升车内乘员舒适性，减少乘客受惊感。
2. **如何防范幽灵刹车？**
   单纯依靠单目前向雷达极易将高架金属接缝误判为加塞障碍物。量产自动驾驶系统通常采用**毫米波雷达 + 视觉摄像头融合**：摄像头识别车辆变道意图与横向车速，毫米波雷达确认纵向距离，满足双重条件才允许 AEB 介入。
3. **分级制动架构**：
   本实验演示的是“一脚刹停”（直接触发 $-5.0 m/s^2$）。在量产中，通常采用三级阶梯控制：
   $$\text{声光预警 (FCW)} \longrightarrow \text{预制动点刹 (-3.0 } m/s^2) \longrightarrow \text{紧急全力制动 (-8.0 } m/s^2)$$
   这正是我们在下一个实验 **A3 (AEB 增强闭环)** 中将要深入攻关的内容！

---

### 🏆 进阶挑战（选做）
1. **舒适性提升**：修改 `MyAEBController`，实现两级分级制动逻辑（当 $2.5s < TTC \le 3.0s$ 时轻微点刹减速 $-2.5 m/s^2$；当 $TTC \le 2.0s$ 时才全力踩死 $-6.0 m/s^2$），观察速度曲线是否更平缓。
2. **考虑前车减速**：当前 TTC 假设前车匀速，若加塞车辆切入后突然急刹，传统 TTC 判定会滞后，请尝试引入目标车加速度 $a_{target}$ 修正相对加速度。

---
## 下一个实验 | Next →
**[A2_acc_follow](./A2_acc_follow.ipynb)** （自适应巡航控制与前车跟随）


### 7.2 工业级 OpenSCENARIO (.xosc) 互操作导出 | Industrial Standards

想把你在本实验中调优的 Cut-in 加塞测试场景迁移到 **CARLA / CarMaker / VTD / AirSim** 等工业仿真软件中？
运行下方单元格，系统将自动生成符合 **ASAM OpenSCENARIO 1.1** 规范的 XML 场景描述文件，并提供一键下载：

In [ ]:
from common.xosc_converter import create_cutin_scenario_xosc, validate_xosc_xml, render_xosc_download_button
from IPython.display import HTML, display

# 生成标准化场景 XML 并校验合规性
custom_xosc = create_cutin_scenario_xosc(
    ego_speed_kmh=50.4,       # 自车巡航速度 (14 m/s)
    target_speed_kmh=43.2,    # 加塞切入速度 (12 m/s)
    initial_gap_m=25.0,       # 初始切入时距 (m)
    cutin_time_s=3.5,         # 切入时机 (s)
)

is_valid, _ = validate_xosc_xml(custom_xosc)
if is_valid:
    display(HTML(render_xosc_download_button(custom_xosc, filename="cutin_iso22179.xosc")))
